# Stand-alone notebook for registering the FlyWire circuit

Adapted from `circuits_hardcoded/register_circuit.ipynb` for the FlyWire
*Drosophila* whole-brain **point-neuron** model.

- Checks metadata provided
- Builds the SONATA circuit (if not already present)
- Registers the circuit entity and uploads assets

Registration logic is reused verbatim from
`circuits_hardcoded/register_circuit_helpers.py`.

## Circuit information

General metadata for the FlyWire model. **Linked entities (`species`,
`subject`, `brain_region`, `license`) must already exist in entitycore** —
the helper functions raise if they are not found.

In [ ]:
circuit_metadata = {
    # Name of the circuit
    "name": "FlyWire Test",

    # Description of the circuit
    "description": (
        "FlyWire whole-brain Drosophila point-neuron model (SONATA), "
        "converted with examples/J_drosophila/drosophila_to_brian2_sonata.py"
    ),

    # Scale of the circuit (single, pair, small, microcircuit, region, system, whole_brain)
    "scale": "whole_brain",

    # Optional: Name of the root circuit (top parent in the derivation hierarchy), if any
    "root": None,

    # Optional: Name of the parent circuit (direct parent it was extracted/derived from)
    "parent": None,

    # Optional: Type of circuit derivation (circuit_extraction, circuit_rewiring), if parent given
    "derivation_type": None,

    # # Species name [Must exist in entitycore; otherwise, must first be registered]
    # "species": "Drosophila melanogaster",

    # # Subject name [Must exist in entitycore; its species must equal `species` above].
    # # Set this to the exact registered Drosophila subject name in your entitycore.
    # "subject": "Drosophila melanogaster",

    # Species name [Must exist in entitycore; otherwise, must first be registered]
    "species": "Rattus norvegicus",

    # Subject name [Must exist in entitycore; otherwise, must first be registered]
    "subject": "Average rat P14",

    # Brain region name [Must exist in entitycore; otherwise, must first be registered].
    # Set this to the exact registered whole-brain region name in your entitycore.
    "brain_region": "Primary somatosensory area",

    # Optional: License name [Must exist in entitycore if not None]
    "license": None,

    # Optional: Short, human readable publication string
    "published_in": None,

    # Optional: Contact e-mail address
    "contact": None,

    # Optional: Experiment date ("November, 2024" -> 01.11.2024, or "27.03.2024")
    "experiment_date": None,

    # Circuit build category (computational_model, em_reconstruction).
    # LIF point-neuron simulation model -> computational_model.
    "build_category": "computational_model",

    # Properties of the circuit (point-neuron model: no morphologies/me-models/spines)
    "has_morphologies": False,
    "has_point_neurons": True,
    "has_electrical_cell_models": False,
    "has_spines": False,
    

    # Counts of the FlyWire SONATA circuit produced by drosophila_to_brian2_sonata.py
    "number_neurons": 127400,      # intrinsic neurons
    "number_synapses": 14687178,   # intrinsic synapses (one per connected pair)
    "number_connections": 14687178,  # intrinsic connections (aggregated FlyWire edges)
    }

### Optional: Publications

DOI -> {"type": entity_source | component_source | application}. Each DOI must
already be registered in entitycore. Empty for *FlyWire Test*.

In [2]:
circuit_publications = {}

### Optional: Contributors

Agent name -> {"type": person | organization | consortium, "role": ...}. Each
agent must already exist in entitycore. Empty for *FlyWire Test*.

In [3]:
circuit_contributions = {}

### Circuit file paths

The original notebook points `circuit_path` at a pre-existing SONATA folder.
Here the FlyWire SONATA circuit is built locally (if not already present) with
`drosophila_to_brian2_sonata.convert()`, and `circuit_path` is set to that
folder (it must contain `circuit_config.json` and `node_sets.json`).

In [4]:
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()
REPO_ROOT = HERE.parents[1]                     # .../obi-one
OBI_MAIN = HERE.parents[2]                      # .../obi-main
J_DROSOPHILA = REPO_ROOT / "examples" / "J_drosophila"
CIRCUIT_DIR = HERE / "sonata_circuit"
DROSOPHILA_REPO = HERE / "Drosophila_brain_model"

if not (CIRCUIT_DIR / "circuit_config.json").exists():
    if not DROSOPHILA_REPO.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/philshiu/Drosophila_brain_model.git",
            str(DROSOPHILA_REPO),
        ])
    for p in (J_DROSOPHILA, DROSOPHILA_REPO):
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
    import drosophila_to_brian2_sonata as dros_sonata
    from model import default_params

    SUGAR_NODES = [
        720575940611875570, 720575940612670570, 720575940616885538,
        720575940617000768, 720575940617937543, 720575940620900446,
        720575940621502051, 720575940621754367, 720575940624963786,
        720575940628853239, 720575940629176663, 720575940630233916,
        720575940630797113, 720575940632425919, 720575940632889389,
        720575940633143833, 720575940637568838, 720575940638202345,
        720575940639198653, 720575940639332736, 720575940640649691,
    ]
    dros_sonata.convert(
        CIRCUIT_DIR,
        path_comp=DROSOPHILA_REPO / "2023_03_23_completeness_630_final.csv",
        path_connnections=DROSOPHILA_REPO / "2023_03_23_connectivity_630_final.parquet",
        sugar_nodes=SUGAR_NODES,
        default_params=default_params,
    )

# Mandatory: SONATA circuit folder containing circuit_config.json
circuit_path = str(CIRCUIT_DIR)

# Optional assets (registered later if not provided)
main_figure_path = None          # Main figure (.webp) for the platform
sim_designer_figure_path = None  # Sim designer figure (.png)
compressed_circuit_path = None   # Compressed SONATA circuit (.gz)
matrix_path = None               # Connectivity matrix folder (matrix_config.json)
node_stats_figure_path = None
network_stats_a_figure_path = None
network_stats_b_figure_path = None

print(f"circuit_path: {circuit_path}")

circuit_path: /Users/james/Documents/obi/code/obi-main/obi-one/examples/J_drosophila_brian2_sonata/sonata_circuit


## Get DB client

Connects to **staging**. `obi_auth` caches the last sign-in on disk and
`get_token()` silently reuses it, so the cached staging token is cleared first
to pick up a freshly signed-in account.

In [5]:
import obi_one as obi
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token
from obi_auth.config import settings as obi_auth_settings
from obi_auth.storage import Storage as ObiAuthStorage
from obi_auth.typedef import DeploymentEnvironment

ObiAuthStorage(
    config_dir=obi_auth_settings.config_dir,
    environment=DeploymentEnvironment.staging,
).clear()

# Staging "Shared results + tests" project
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("connected to staging entitycore")

connected to staging entitycore


## Load the registration helpers

`register_circuit_helpers.py` lives in the sibling `circuits_hardcoded`
repository; add it to `sys.path` so its functions can be reused unchanged.

In [6]:
CIRCUITS_HARDCODED = OBI_MAIN / "circuits_hardcoded"
assert CIRCUITS_HARDCODED.is_dir(), f"{CIRCUITS_HARDCODED} not found"
if str(CIRCUITS_HARDCODED) not in sys.path:
    sys.path.insert(0, str(CIRCUITS_HARDCODED))

from register_circuit_helpers import (
    check_counts,
    check_if_circuit_exists,
    get_brain_region,
    get_contributions,
    get_exp_date,
    get_license,
    get_parent_circuit,
    get_publications,
    get_root_circuit,
    get_subject,
    register_asset,
    register_circuit_entity,
    register_contributions,
    register_derivation,
    register_publication_links,
)
print(f"helpers loaded from {CIRCUITS_HARDCODED}")

helpers loaded from /Users/james/Documents/obi/code/obi-main/circuits_hardcoded


## Check provided metadata and dependencies

### Check if circuit is registered in entitycore already

The circuit must not exist in entitycore yet!

In [7]:
check_if_circuit_exists(client, circuit_metadata)

Circuit 'FlyWire Test' not yet registered.


### Check and retrieve dependencies

All dependencies must already exist in entitycore!

In [8]:
# Check/get subject
subject = get_subject(client, circuit_metadata)

# Check/get brain region
brain_region = get_brain_region(client, circuit_metadata)

# Check/get license
license = get_license(client, circuit_metadata)

# Check/get root circuit
root = get_root_circuit(client, circuit_metadata)

# Check/get parent circuit [Note: Parent = Derivation of type <derivation_type>]
parent = get_parent_circuit(client, circuit_metadata)

# Check neuron/synapse/connection counts
check_counts(circuit_metadata)

# Determine experiment date
exp_date = get_exp_date(circuit_metadata)

# Check/get contributing agents
contribution_dict = get_contributions(client, circuit_contributions, verbose=False)

# Check/get publications
publication_dict = get_publications(client, circuit_publications, verbose=False)

Subject 'Average rat P14' (ID e5ecb660-504f-4840-b674-f31f0eada439)
Brain region 'Primary somatosensory area' (ID 61d89b07-dfa0-439a-9187-7ebfe60e212b)
Root circuit: None
Parent circuit: None
#Neurons: 127400, #Synapses: 14687178, #Connections: 14687178, Scale: whole_brain
Experiment date: None
Contributors: {}
Publications: {}


## Register circuit entity, assets, and links

- Keep `check_only = True` for a dry run without actual registration first!
- Use `make_public` to register as a public entity

In [13]:
check_only = False   # If True, only check, no actual registration
make_public = False  # Register as public entity

### Register circuit entity

In [14]:
registered_circuit = register_circuit_entity(
    client, circuit_metadata, subject, brain_region, license, root, exp_date,
    make_public=make_public, check_only=check_only,
)

Circuit entity 'FlyWire Test': ID dc57f86d-a591-4d7e-a65d-54e097e8a021


### Register assets
- SONATA circuit folder
- Optional: Overview / sim designer image
- Optional: Compressed SONATA circuit
- Optional: Connectivity matrix folder
- Optional: Basic connectivity figures

In [15]:
# Register circuit folder asset
circuit_folder_asset = register_asset(
    client, file_path=circuit_path, asset_label="sonata_circuit",
    registered_circuit=registered_circuit, check_only=check_only,
)

# Compressed circuit asset
gz_asset = register_asset(
    client, file_path=compressed_circuit_path,
    asset_label="compressed_sonata_circuit",
    registered_circuit=registered_circuit, check_only=check_only,
)

# Connectivity matrix folder asset
matrix_folder_asset = register_asset(
    client, file_path=matrix_path, asset_label="circuit_connectivity_matrices",
    registered_circuit=registered_circuit, check_only=check_only,
)

# Main circuit figure for detailed explore page
main_fig_asset = register_asset(
    client, file_path=main_figure_path, asset_label="circuit_visualization",
    registered_circuit=registered_circuit, check_only=check_only,
)

# Circuit figure for simulation designer
sim_fig_asset = register_asset(
    client, file_path=sim_designer_figure_path,
    asset_label="simulation_designer_image",
    registered_circuit=registered_circuit, check_only=check_only,
)

# Basic connectivity plots
node_stats_fig_asset = register_asset(
    client, file_path=node_stats_figure_path, asset_label="node_stats",
    registered_circuit=registered_circuit, check_only=check_only,
)
network_stats_a_fig_asset = register_asset(
    client, file_path=network_stats_a_figure_path, asset_label="network_stats_a",
    registered_circuit=registered_circuit, check_only=check_only,
)
network_stats_b_fig_asset = register_asset(
    client, file_path=network_stats_b_figure_path, asset_label="network_stats_b",
    registered_circuit=registered_circuit, check_only=check_only,
)

assetID (sonata_circuit): c1c71800-48f8-48ff-8814-e991cc792e3f
INFO: No path for 'compressed_sonata_circuit' asset provided - SKIPPING
INFO: No path for 'circuit_connectivity_matrices' asset provided - SKIPPING
INFO: No path for 'circuit_visualization' asset provided - SKIPPING
INFO: No path for 'simulation_designer_image' asset provided - SKIPPING
INFO: No path for 'node_stats' asset provided - SKIPPING
INFO: No path for 'network_stats_a' asset provided - SKIPPING
INFO: No path for 'network_stats_b' asset provided - SKIPPING


### Register entity links
- Derivation links (if any)
- Contribution links (if any)
- Publication links (if any)

In [16]:
# Derivation links
derivation = register_derivation(
    client, from_entity=parent,
    derivation_type=circuit_metadata.get("derivation_type"),
    registered_circuit=registered_circuit, check_only=check_only,
)

# Contribution links
contributions = register_contributions(
    client, contribution_dict=contribution_dict,
    registered_circuit=registered_circuit, check_only=check_only,
)

# Publication links
publ_links = register_publication_links(
    client, publication_dict=publication_dict,
    registered_circuit=registered_circuit, check_only=check_only,
)

INFO: No derivation parent provided - SKIPPING
Contributions: 0 registered
Publication links: 0 registered
